# 03 — Model Training: TSA Checkpoint Passenger Flow
**Authors:** Arend Colle, Vivek Sriram, Tiffany T. Nguyen

**Problem:** TSA staffing and hiring decisions are made roughly a year in advance.
This model predicts hourly checkpoint throughput using only information available at
planning time: published airline schedules, historical load factors, and the calendar.

**Inputs:** TSA throughput CSVs · BTS on-time CSVs · BTS T-100 segment CSVs
**Output:** `models/lgbmModel.pkl`

## 1. Imports and Constants

In [13]:
import os, gc, glob, warnings, joblib
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
import holidays, lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore', category=UserWarning)

ROOT            = Path(r'C:/Users/vivek/OneDrive/Desktop/ml project')
THROUGHPUT_GLOB = str(ROOT / 'throuput data' / 'TsaThroughput.*.csv')
ONTIME_GLOB     = str(ROOT / 'ontime data'   / 'ontime_*.csv')
BTS_GLOB        = str(ROOT / 'bts data'      / '*.csv')
CACHE_DIR       = ROOT / 'cached_filtered'
MODELS_DIR      = ROOT / 'models'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_AIRPORTS = [
    'ATL', 'LAX', 'DFW', 'DEN', 'ORD', 'JFK', 'MCO', 'LAS', 'CLT', 'MIA',
    'PHX', 'IAH', 'BOS', 'FLL', 'MSP', 'LGA', 'DTW', 'SEA', 'SFO', 'EWR'
]
TEST_DAYS     = 60
VAL_DAYS      = 60
MODELDF_CACHE = str(CACHE_DIR / 'modelDf.csv')

print(f'Throughput files : {len(glob.glob(THROUGHPUT_GLOB))}')
print(f'On-time files    : {len(glob.glob(ONTIME_GLOB))}')
print(f'BTS files        : {len(glob.glob(BTS_GLOB))}')

Throughput files : 20
On-time files    : 4
BTS files        : 4


## 2. Build and Cache Model DataFrame


In [14]:
FORCE_REBUILD = True

if FORCE_REBUILD and os.path.exists(MODELDF_CACHE):
    os.remove(MODELDF_CACHE)
    print('Stale cache deleted — rebuilding.')

print('Cache ready.' if os.path.exists(MODELDF_CACHE) else 'No cache found — run Sections 2a-2d.')

Stale cache deleted — rebuilding.
No cache found — run Sections 2a-2d.


## 2a. Load TSA Throughput
One file at a time to keep peak memory low. Wide checkpoint columns melted to long format.

In [15]:
def loadThroughput(globPattern, airports):
    parts = []
    for f in sorted(glob.glob(globPattern)):
        try: raw = pd.read_csv(f)
        except Exception as e: print(f'  Skipping {f}: {e}'); continue

        idCols    = [c for c in ['Date', 'Hour'] if c in raw.columns]
        valueCols = [c for c in raw.columns if c not in idCols and c[:3] in airports]
        if not valueCols: continue

        parts.append(
            raw[idCols + valueCols]
            .melt(id_vars=idCols, var_name='checkpointCol', value_name='totalPassengers')
            .dropna(subset=['totalPassengers'])
        )
        del raw; gc.collect()

    if not parts: return pd.DataFrame()

    df = pd.concat(parts, ignore_index=True)
    df['date']            = pd.to_datetime(df['Date'], errors='coerce')
    df['hour']            = pd.to_datetime(df['Hour'], format='%H:%M:%S', errors='coerce').dt.hour.astype('int8')
    df['airportCode']     = df['checkpointCol'].str[:3]
    df['checkpointName']  = df['checkpointCol'].str[4:].str.strip()
    df['totalPassengers'] = pd.to_numeric(df['totalPassengers'], errors='coerce')
    return (df.drop(columns=['Date', 'Hour', 'checkpointCol'])
              .loc[lambda d: d['airportCode'].isin(airports)]
              .dropna(subset=['date', 'hour', 'totalPassengers'])
              .reset_index(drop=True))


throughputDf = loadThroughput(THROUGHPUT_GLOB, TARGET_AIRPORTS)
# On-time data only available from 2022 — drop earlier rows
throughputDf = throughputDf[throughputDf['date'].dt.year >= 2022].reset_index(drop=True)
print(f'Throughput: {len(throughputDf):,} rows ({throughputDf["date"].dt.year.min()}-{throughputDf["date"].dt.year.max()})')
throughputDf.head()

Throughput: 3,159,423 rows (2022-2025)


,totalPassengers,date,hour,airportCode,checkpointName
0,50.0,2022-01-01,16,ATL,E Concourse Checkpoint
1,379.0,2022-01-01,17,ATL,E Concourse Checkpoint
2,504.0,2022-01-01,18,ATL,E Concourse Checkpoint
3,902.0,2022-01-01,19,ATL,E Concourse Checkpoint
4,300.0,2022-01-01,20,ATL,E Concourse Checkpoint


## 2b. Load BTS On-Time Performance
Aggregated to hourly departure counts per airport. 

In [16]:
def loadOntime(globPattern, airports):
    COLS = ['FL_DATE', 'ORIGIN', 'CRS_DEP_TIME', 'DEP_TIME', 'CANCELLED']

    def extractHour(x):
        try: return int(str(int(x)).zfill(4)[:2])
        except: return np.nan

    aggParts = []
    for f in sorted(glob.glob(globPattern)):
        try: raw = pd.read_csv(f, low_memory=False, usecols=lambda c: c.upper() in COLS)
        except Exception as e: print(f'  Skipping {f}: {e}'); continue

        raw.columns = [c.upper() for c in raw.columns]
        timeCol = next((c for c in ('CRS_DEP_TIME', 'DEP_TIME') if c in raw.columns), None)
        if not {'FL_DATE', 'ORIGIN'}.issubset(raw.columns) or not timeCol: continue

        df = raw[raw['ORIGIN'].isin(airports)].copy()
        if 'CANCELLED' in df.columns: df = df[df['CANCELLED'] < 1]
        df['date'] = pd.to_datetime(df['FL_DATE'], format='mixed', errors='coerce')
        df['hour'] = df[timeCol].apply(extractHour)
        df = df.dropna(subset=['date', 'hour'])
        df['hour'] = df['hour'].astype('int8')

        aggParts.append(
            df.groupby(['ORIGIN', 'date', 'hour']).size()
              .reset_index(name='hourlyDepartureCount')
              .rename(columns={'ORIGIN': 'airportCode'})
        )
        del raw, df; gc.collect()

    if not aggParts: return pd.DataFrame()
    result = (pd.concat(aggParts, ignore_index=True)
                .groupby(['airportCode', 'date', 'hour'], as_index=False)['hourlyDepartureCount'].sum())
    print(f'On-time: {len(result):,} airport-date-hour rows')
    return result


departuresDf = loadOntime(ONTIME_GLOB, TARGET_AIRPORTS)
departuresDf.head()

On-time: 604,683 airport-date-hour rows


,airportCode,date,hour,hourlyDepartureCount
0,ATL,2022-01-01,0,4
1,ATL,2022-01-01,1,2
2,ATL,2022-01-01,2,2
3,ATL,2022-01-01,5,4
4,ATL,2022-01-01,6,14


## 2c. Load BTS T-100 Load Factors
Provides avg seats per flight and avg load factor by airport-month 

In [5]:
def loadLoadFactors(globPattern, airports):
    # expectedPassengerVolume = hourlyDepartureCount × avgSeatsPerFlight × avgLoadFactor
    COLS = ['ORIGIN', 'PASSENGERS', 'SEATS', 'DEPARTURES_PERFORMED', 'MONTH']
    parts = []
    for f in glob.glob(globPattern):
        try:
            raw = pd.read_csv(f, usecols=lambda c: c.upper() in COLS)
            raw.columns = [c.upper() for c in raw.columns]
            parts.append(raw[raw['ORIGIN'].isin(airports) & (raw['SEATS'] > 0) & (raw['DEPARTURES_PERFORMED'] > 0)])
        except Exception as e: print(f'  Skipping {f}: {e}')

    if not parts: return pd.DataFrame()
    df = pd.concat(parts, ignore_index=True)
    df['MONTH']       = df['MONTH'].astype(int)
    df['loadFactor']  = df['PASSENGERS'] / df['SEATS']
    df['seatsPerDep'] = df['SEATS']      / df['DEPARTURES_PERFORMED']

    return (df.groupby(['ORIGIN', 'MONTH'], as_index=False)
               .agg(avgLoadFactor=('loadFactor', 'mean'), avgSeatsPerFlight=('seatsPerDep', 'mean'))
               .rename(columns={'ORIGIN': 'airportCode', 'MONTH': 'month'}))


loadFactorsDf = loadLoadFactors(BTS_GLOB, TARGET_AIRPORTS)
print(f'T-100: {len(loadFactorsDf):,} airport-month combos')
loadFactorsDf.head()

T-100: 240 airport-month combos


,airportCode,month,avgLoadFactor,avgSeatsPerFlight
0,ATL,1,0.702881,153.868241
1,ATL,2,0.720940,154.494535
2,ATL,3,0.770105,156.377092
3,ATL,4,0.783703,155.645258
4,ATL,5,0.822087,158.531437


## 2d. Merge, Feature Engineering, and Cache

In [6]:
def addCalendarFeatures(df):
    """Adds dayOfWeek, month, isWeekend, isHoliday, daysToNearestHoliday."""
    usHolidays   = holidays.US(years=range(2022, 2026))
    holidayDates = set(usHolidays.keys())
    df['dayOfWeek'] = df['date'].dt.weekday
    df['month']     = df['date'].dt.month
    df['isWeekend'] = df['dayOfWeek'].isin([5, 6]).astype('int8')
    df['isHoliday'] = df['date'].dt.date.apply(lambda d: int(d in holidayDates)).astype('int8')
    sortedHols = np.array(sorted(holidayDates), dtype='datetime64[D]')
    dateVals   = df['date'].values.astype('datetime64[D]')
    idxR = np.clip(np.searchsorted(sortedHols, dateVals),     0, len(sortedHols) - 1)
    idxL = np.clip(np.searchsorted(sortedHols, dateVals) - 1, 0, len(sortedHols) - 1)
    df['daysToNearestHoliday'] = np.minimum(
        np.abs((dateVals - sortedHols[idxL]).astype(int)),
        np.abs((dateVals - sortedHols[idxR]).astype(int))
    ).astype('int16')
    return df


modelDf = throughputDf.copy()
del throughputDf; gc.collect()

if not departuresDf.empty:
    modelDf = modelDf.merge(departuresDf, how='left', on=['airportCode', 'date', 'hour'])
del departuresDf; gc.collect()

modelDf['month'] = modelDf['date'].dt.month
if not loadFactorsDf.empty:
    modelDf = modelDf.merge(loadFactorsDf, how='left', on=['airportCode', 'month'])
del loadFactorsDf; gc.collect()

# Hours with no departure record = no scheduled flights that hour
modelDf['hourlyDepartureCount'] = modelDf['hourlyDepartureCount'].fillna(0)
modelDf['expectedPassengerVolume'] = (
    modelDf['hourlyDepartureCount'] * modelDf['avgSeatsPerFlight'] * modelDf['avgLoadFactor']
).astype('float32')

modelDf = addCalendarFeatures(modelDf)
modelDf = modelDf.dropna(subset=['totalPassengers']).reset_index(drop=True)

for col in ['hourlyDepartureCount', 'avgLoadFactor', 'avgSeatsPerFlight', 'totalPassengers']:
    modelDf[col] = modelDf[col].astype('float32')

modelDf.to_csv(MODELDF_CACHE, index=False)
print(f'Cached {len(modelDf):,} rows | Memory: {modelDf.memory_usage(deep=True).sum() / 1e6:.1f} MB')
modelDf.head()

Cached 3,159,423 rows | Memory: 480.9 MB


,totalPassengers,date,hour,airportCode,checkpointName,hourlyDepartureCount,month,avgLoadFactor,avgSeatsPerFlight,expectedPassengerVolume,dayOfWeek,isWeekend,isHoliday,daysToNearestHoliday
0,50.0,2022-01-01,16,ATL,E Concourse Checkpoint,22.0,1,0.702881,153.86824,2379.324951,5,1,1,0
1,379.0,2022-01-01,17,ATL,E Concourse Checkpoint,37.0,1,0.702881,153.86824,4001.591797,5,1,1,0
2,504.0,2022-01-01,18,ATL,E Concourse Checkpoint,26.0,1,0.702881,153.86824,2811.929443,5,1,1,0
3,902.0,2022-01-01,19,ATL,E Concourse Checkpoint,36.0,1,0.702881,153.86824,3893.440674,5,1,1,0
4,300.0,2022-01-01,20,ATL,E Concourse Checkpoint,30.0,1,0.702881,153.86824,3244.533936,5,1,1,0


## 3. Load Cached Data
**Start here on subsequent runs.**

In [7]:
modelDf = pd.read_csv(MODELDF_CACHE, parse_dates=['date']).sort_values('date').reset_index(drop=True)
print(f'Loaded {len(modelDf):,} rows | {modelDf["date"].min().date()} -> {modelDf["date"].max().date()}')
modelDf.head()

Loaded 3,159,423 rows | 2022-01-01 -> 2025-05-31


,totalPassengers,date,hour,airportCode,checkpointName,hourlyDepartureCount,month,avgLoadFactor,avgSeatsPerFlight,expectedPassengerVolume,dayOfWeek,isWeekend,isHoliday,daysToNearestHoliday
0,50.0,2022-01-01,16,ATL,E Concourse Checkpoint,22.0,1,0.702881,153.86824,2379.3250,5,1,1,0
1,187.0,2022-01-01,11,CLT,B Checkpoint,45.0,1,0.697046,125.51717,3937.1055,5,1,1,0
2,249.0,2022-01-01,12,CLT,B Checkpoint,15.0,1,0.697046,125.51717,1312.3685,5,1,1,0
3,306.0,2022-01-01,13,CLT,B Checkpoint,29.0,1,0.697046,125.51717,2537.2456,5,1,1,0
4,237.0,2022-01-01,14,CLT,B Checkpoint,37.0,1,0.697046,125.51717,3237.1755,5,1,1,0


## 4. Feature Matrix
Categoricals label-encoded — LightGBM handles these correctly via tree splits, unlike linear models.

In [8]:
FEATURES = [
    'hour', 'airportCode_lbl', 'checkpointName_lbl',
    'hourlyDepartureCount', 'avgSeatsPerFlight', 'avgLoadFactor', 'expectedPassengerVolume',
    'dayOfWeek', 'month', 'isWeekend', 'isHoliday', 'daysToNearestHoliday'
]
TARGET = 'totalPassengers'

leAirport, leCheckpoint = LabelEncoder(), LabelEncoder()
modelDf['airportCode_lbl']    = leAirport.fit_transform(modelDf['airportCode']).astype('int8')
modelDf['checkpointName_lbl'] = leCheckpoint.fit_transform(modelDf['checkpointName'].astype(str)).astype('int16')

X = modelDf[FEATURES].fillna(0).astype({
    'hour': 'int8', 'airportCode_lbl': 'int8', 'checkpointName_lbl': 'int16',
    'dayOfWeek': 'int8', 'month': 'int8', 'isWeekend': 'int8', 'isHoliday': 'int8',
    'daysToNearestHoliday': 'int16', 'hourlyDepartureCount': 'float32',
    'avgSeatsPerFlight': 'float32', 'avgLoadFactor': 'float32', 'expectedPassengerVolume': 'float32',
})
y = modelDf[TARGET].astype('float32')
print(f'Feature matrix: {X.shape[0]:,} rows x {X.shape[1]} features')

Feature matrix: 3,159,423 rows x 12 features


## 5. Temporal Train / Validation / Test Split
Chronological split — no future data leaks into training.
Mirrors the staffing scenario: model trained on past actuals, evaluated on a future holdout period.

In [9]:
maxDate   = modelDf['date'].max()
testStart = maxDate   - pd.Timedelta(TEST_DAYS, 'd')
valStart  = testStart - pd.Timedelta(VAL_DAYS,  'd')

trainMask = modelDf['date'] < valStart
valMask   = (modelDf['date'] >= valStart) & (modelDf['date'] < testStart)
testMask  = modelDf['date'] >= testStart

xTrain, yTrain = X[trainMask].copy(), y[trainMask].copy()
xVal,   yVal   = X[valMask].copy(),   y[valMask].copy()
xTest,  yTest  = X[testMask].copy(),  y[testMask].copy()

testDatesDf = modelDf.loc[testMask, ['date', TARGET]].copy().reset_index(drop=True)
# trainBase used for naive baseline — includes hour and dayOfWeek for the lookup
trainBase   = modelDf.loc[trainMask, ['airportCode', 'checkpointName', 'hour', 'dayOfWeek', 'month', TARGET]].copy()
del modelDf, X, y; gc.collect()

print(f'Train: {len(yTrain):,}  |  Val: {len(yVal):,}  |  Test: {len(yTest):,}')

Train: 2,845,301  |  Val: 155,101  |  Test: 159,021


## 6. Naive Baseline (Year-Ahead Staffing Benchmark)
Simulates a planner predicting throughput a year out using only multi-year historical averages.
For a given slot, e.g. ATL E Concourse, Thursday 1pm in March, the prediction is the mean
throughput across all matching rows in the training data (2022-2024).
No real-time or operationally-observed information is assumed.

In [ ]:
# Group by checkpoint + hour + day-of-week + month 
lookupCols = ['airportCode', 'checkpointName', 'hour', 'dayOfWeek', 'month']
meanLookup = (
    trainBase.groupby(lookupCols)[TARGET].mean()
             .reset_index().rename(columns={TARGET: 'naivePred'})
)

testLookup = (
    pd.read_csv(MODELDF_CACHE, usecols=['date', *lookupCols, TARGET], parse_dates=['date'])
      .sort_values('date').reset_index(drop=True)
)
testLookup = testLookup[testLookup['date'] >= testLookup['date'].max() - pd.Timedelta(TEST_DAYS, 'd')]
testLookup = testLookup.merge(meanLookup, how='left', on=lookupCols)
testLookup['naivePred'] = testLookup['naivePred'].fillna(yTrain.mean())

naiveR2   = r2_score(testLookup[TARGET], testLookup['naivePred'])
naiveMae  = mean_absolute_error(testLookup[TARGET], testLookup['naivePred'])
naiveRmse = np.sqrt(mean_squared_error(testLookup[TARGET], testLookup['naivePred']))
del testLookup, meanLookup, trainBase; gc.collect()

print(f'Naive Baseline  R2: {naiveR2:.4f}  MAE: {naiveMae:.2f}  RMSE: {naiveRmse:.2f}')

Naive Baseline  R2: 0.7968  MAE: 147.07  RMSE: 213.53


## 7. Train LightGBM
Gradient boosted trees iteratively correct errors from the naive historical average.
Features like hourly departure counts and holiday proximity give it signal the baseline cannot use.
Early stopping determines the optimal number of rounds from the validation set.

In [17]:
lgbTrain = lgb.Dataset(xTrain, label=yTrain, free_raw_data=False)
lgbVal   = lgb.Dataset(xVal,   label=yVal, reference=lgbTrain, free_raw_data=False)

params = {
    'objective':         'regression',
    'metric':            'rmse',
    'learning_rate':     0.05,
    'num_leaves':        127,
    'min_child_samples': 50,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'verbose':           -1,
}

lgbModel = lgb.train(
    params, lgbTrain,
    num_boost_round=10000,
    valid_sets=[lgbVal],
    callbacks=[lgb.early_stopping(100, verbose=True), lgb.log_evaluation(100)]
)

yValPred = lgbModel.predict(xVal)
print(f'Validation  R2: {r2_score(yVal, yValPred):.4f}  MAE: {mean_absolute_error(yVal, yValPred):.2f}')
del lgbTrain, lgbVal; gc.collect()

Training until validation scores don't improve for 100 rounds
[100]	valid_0's rmse: 204.608


KeyboardInterrupt: 

## 8. Test Set Evaluation
All metrics computed on the held-out test period — never seen during training or tuning.

In [12]:
yTestPred = lgbModel.predict(xTest)

testR2   = r2_score(yTest, yTestPred)
testMae  = mean_absolute_error(yTest, yTestPred)
testRmse = np.sqrt(mean_squared_error(yTest, yTestPred))

# Daily Pearson r: correlation between predicted and actual daily totals summed across all checkpoints
# Follows Monmousseau et al. benchmarking methodology
testDatesDf['pred'] = yTestPred
dailyAgg = testDatesDf.groupby(testDatesDf['date'].dt.date).agg(
    actualTotal=(TARGET, 'sum'), predTotal=('pred', 'sum')
)
dailyPearson = pearsonr(dailyAgg['actualTotal'], dailyAgg['predTotal'])[0] if len(dailyAgg) > 1 else float('nan')
del xTest, yTest, yTestPred, testDatesDf, dailyAgg; gc.collect()

print(f'LightGBM  R2: {testR2:.4f}  MAE: {testMae:.2f}  RMSE: {testRmse:.2f}  Pearson r: {dailyPearson:.4f}')

LightGBM  R2: 0.8509  MAE: 123.85  RMSE: 182.93  Pearson r: 0.9398


## 9. Results and Feature Importance

In [13]:
summaryDf = pd.DataFrame({
    'R²':               [f'{naiveR2:.4f}',   f'{testR2:.4f}'],
    'MAE (passengers)': [f'{naiveMae:.2f}',  f'{testMae:.2f}'],
    'RMSE':             [f'{naiveRmse:.2f}', f'{testRmse:.2f}'],
    'Daily Pearson r':  ['—',                f'{dailyPearson:.4f}'],
}, index=['Naive Baseline', 'LightGBM']).T
print(summaryDf.to_string())
print()

# Gain importance: how much each feature reduces prediction error across all tree splits
importanceDf = (
    pd.DataFrame({'feature':    lgbModel.feature_name(),
                  'importance': lgbModel.feature_importance(importance_type='gain')})
      .sort_values('importance', ascending=False)
      .reset_index(drop=True)
)
print(importanceDf.to_string(index=False))

                 Naive Baseline LightGBM
R²                       0.7968   0.8509
MAE (passengers)         147.07   123.85
RMSE                     213.53   182.93
Daily Pearson r               —   0.9398

                feature   importance
     checkpointName_lbl 2.291608e+12
                   hour 8.409325e+11
        airportCode_lbl 6.774748e+11
      avgSeatsPerFlight 2.216858e+11
expectedPassengerVolume 9.563532e+10
   hourlyDepartureCount 9.070535e+10
              dayOfWeek 5.408621e+10
          avgLoadFactor 5.078015e+10
                  month 3.710118e+10
   daysToNearestHoliday 2.108342e+10
              isHoliday 2.099031e+09
              isWeekend 1.807776e+09


## 10. Save Model Artifact

In [14]:
artifactPath = str(MODELS_DIR / 'lgbmModel.pkl')
joblib.dump({
    'model':        lgbModel,
    'leAirport':    leAirport,
    'leCheckpoint': leCheckpoint,
    'features':     FEATURES,
    'testMetrics':  {'r2': testR2, 'mae': testMae, 'rmse': testRmse, 'dailyPearson': dailyPearson}
}, artifactPath)
print(f'Saved -> {artifactPath}')

Saved -> C:\Users\vivek\OneDrive\Desktop\ml project\models\lgbmModel.pkl


In [ ]:
### MIDPOINT VISUAL 01 - Baseline vs LightGBM metric comparison ###
import matplotlib.pyplot as plt
metrics = ["R2", "MAE (passengers)", "RMSE"]
baseline = [naiveR2, naiveMae, naiveRmse]
lgbm_val = [testR2, testMae, testRmse]
fig, axes = plt.subplots(1, 3, figsize=(10,4))
colors = ["#4C72B0", "#DD8452"]
for ax, metric, bval, lval in zip(axes, metrics, baseline, lgbm_val):
    bars = ax.bar(["Naive Baseline","LightGBM"], [bval, lval], color=colors, width=0.5)
    for bar, val in zip(bars, [bval, lval]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(bval,lval)*0.02, f"{val:.3f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
    ax.set_title(metric, fontsize=13, fontweight="bold")
    ax.set_ylim(0, max(bval,lval)*1.18)
    ax.spines[["top","right"]].set_visible(False)
fig.suptitle("Model Performance: Naive Baseline vs LightGBM (Test Set)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("midpoint_visual_01_model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: midpoint_visual_01_model_comparison.png")

In [ ]:
### MIDPOINT VISUAL 02 - Feature importance top 10 ###
import pandas as pd
imp_df = (pd.DataFrame({"feature": lgbModel.feature_name(), "importance": lgbModel.feature_importance(importance_type="gain")})
           .sort_values("importance", ascending=True).tail(10))
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(imp_df["feature"], imp_df["importance"]/imp_df["importance"].sum()*100, color="#4C72B0", edgecolor="white")
ax.set_xlabel("% of Total Gain", fontsize=12)
ax.set_title("Top 10 Feature Importances (Gain)", fontsize=14, fontweight="bold")
ax.spines[["top","right"]].set_visible(False)
for bar in bars:
    ax.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2, f"{bar.get_width():.1f}%", va="center", fontsize=10)
plt.tight_layout()
plt.savefig("midpoint_visual_02_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: midpoint_visual_02_feature_importance.png")

In [ ]:
### MIDPOINT VISUAL 03 - Actual vs Predicted scatter (val set sample) ###
import numpy as np
yv_pred = lgbModel.predict(xVal)
sample_idx = np.random.default_rng(42).choice(len(yVal), size=min(5000, len(yVal)), replace=False)
act_s = yVal.values[sample_idx]
pred_s = yv_pred[sample_idx]
fig, ax = plt.subplots(figsize=(6,6))
ax.scatter(act_s, pred_s, alpha=0.15, s=8, color="#4C72B0")
lims = [0, max(act_s.max(), pred_s.max())*1.05]
ax.plot(lims, lims, "r--", linewidth=1.5, label="Perfect prediction")
ax.set_xlabel("Actual Passengers", fontsize=12)
ax.set_ylabel("Predicted Passengers", fontsize=12)
ax.set_title("Actual vs Predicted - Validation Set (5k sample)", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("midpoint_visual_03_actual_vs_predicted.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: midpoint_visual_03_actual_vs_predicted.png")

In [ ]:
### MIDPOINT VISUAL 04 - EDA: avg throughput by hour and day of week ###
import pandas as pd
_eda = pd.read_csv(MODELDF_CACHE, usecols=["hour","dayOfWeek","totalPassengers"])
DAY_NAMES = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
hourly_avg = _eda.groupby("hour")["totalPassengers"].mean()
dow_avg = _eda.groupby("dayOfWeek")["totalPassengers"].mean()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,4))
ax1.plot(hourly_avg.index, hourly_avg.values, marker="o", color="#4C72B0", linewidth=2)
ax1.set_xlabel("Hour of Day", fontsize=12)
ax1.set_ylabel("Avg Passengers", fontsize=12)
ax1.set_title("Avg Throughput by Hour", fontsize=13, fontweight="bold")
ax1.spines[["top","right"]].set_visible(False)
ax2.bar([DAY_NAMES[i] for i in dow_avg.index], dow_avg.values, color="#DD8452")
ax2.set_xlabel("Day of Week", fontsize=12)
ax2.set_ylabel("Avg Passengers", fontsize=12)
ax2.set_title("Avg Throughput by Day of Week", fontsize=13, fontweight="bold")
ax2.spines[["top","right"]].set_visible(False)
fig.suptitle("EDA: Passenger Throughput Patterns", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("midpoint_visual_04_eda_patterns.png", dpi=150, bbox_inches="tight")
plt.show()
del _eda
print("Saved: midpoint_visual_04_eda_patterns.png")